In [0]:
# Standard calendar (date dimension) table: 1940-01-01 to 2030-12-31
# Week/day-of-week follow ISO 8601 (weeks start on Monday, day_of_week 1=Mon ... 7=Sun)

target_table = "adb_caresync.silver.calendar"
start_date = "1940-01-01"
end_date = "2030-12-31"

spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Standard calendar date dimension from {start_date} to {end_date}. ISO 8601 weeks (Monday start).'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'{start_date}', DATE'{end_date}', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)              AS datekey,
  d                                                    AS date,
  year(d)                                              AS year,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END            AS half_year,
  quarter(d)                                           AS quarter,
  concat('Q', quarter(d))                              AS quarter_name,
  concat(year(d), '-Q', quarter(d))                    AS year_quarter,
  month(d)                                             AS month,
  date_format(d, 'MMMM')                               AS month_name,
  date_format(d, 'MMM')                                AS month_short_name,
  date_format(d, 'yyyy-MM')                            AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                AS year_month_key,
  weekofyear(d)                                        AS week,
  extract(YEAROFWEEK FROM d)                           AS iso_year,
  concat(extract(YEAROFWEEK FROM d), '-W', lpad(CAST(weekofyear(d) AS STRING), 2, '0')) AS iso_year_week,
  dayofmonth(d)                                        AS day,
  weekday(d) + 1                                       AS day_of_week,
  date_format(d, 'EEEE')                               AS day_name,
  date_format(d, 'EEE')                                AS day_short_name,
  dayofyear(d)                                         AS day_of_year,
  day(last_day(d))                                     AS days_in_month,
  date_sub(d, weekday(d))                              AS week_start_date,
  date_add(d, 6 - weekday(d))                          AS week_end_date,
  trunc(d, 'MM')                                       AS month_start_date,
  last_day(d)                                          AS month_end_date,
  trunc(d, 'QUARTER')                                  AS quarter_start_date,
  date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)      AS quarter_end_date,
  trunc(d, 'YEAR')                                     AS year_start_date,
  make_date(year(d), 12, 31)                           AS year_end_date,
  weekday(d) >= 5                                      AS is_weekend,
  weekday(d) < 5                                       AS is_weekday,
  (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 AS is_leap_year,
  d = trunc(d, 'MM')                                   AS is_month_start,
  d = last_day(d)                                      AS is_month_end,
  d = trunc(d, 'QUARTER')                              AS is_quarter_start,
  d = date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)  AS is_quarter_end,
  d = trunc(d, 'YEAR')                                 AS is_year_start,
  d = make_date(year(d), 12, 31)                       AS is_year_end
FROM dates
""")

display(spark.table(target_table).orderBy("datekey").limit(10))

In [0]:
# Validation: row count, date range, uniqueness of datekey
display(spark.sql(f"""
SELECT COUNT(*)                         AS row_count,
       COUNT(DISTINCT datekey)          AS distinct_datekeys,
       datediff(MAX(date), MIN(date)) + 1 AS expected_days,
       MIN(date)                        AS min_date,
       MAX(date)                        AS max_date
FROM {target_table}
"""))